# Resultados — IDS multiclasse SOME/IP (split temporal)

> **Notebook-artefato do artigo.** Avalia o classificador multiclasse com as **16 features
> comportamentais** (sem cabeçalho) usando **split temporal por arquivo** (honesto, *in-scope*) —
> sem o vazamento do split aleatório. Gera e **salva em `results/`**: tabela por classe,
> **matriz de confusão**, **curvas ROC/PR**, **importância das features**, e as **métricas de
> tempo** (extração de features vs inferência).

Classes: `normal, dos, fuzzy, mitm_single, mitm_multi`. XGBoost: 300 árvores, `max_depth=8`,
`lr=0.3`, `tree_method=hist`.

**Ative a GPU:** Ambiente de execução → Alterar tipo → **T4 GPU** → Executar tudo.


In [ ]:
# Setup: clona o repo + baixa features (LFS) — igual ao 03-params-kim-gpu
import os
REPO = 'someip-ids-multiclass-contentext'
if not os.path.exists('data/ours_ext/X.npz'):
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')
    elif os.path.isdir(REPO):
        os.chdir(REPO)
    else:
        os.system('apt-get -qq install -y git-lfs >/dev/null 2>&1; git lfs install')
        os.system(f'git clone -q https://github.com/GuilhermeFrick/{REPO}.git')
        os.chdir(REPO); os.system('git lfs pull')
if os.path.exists('data/ours_ext/X.npz') and os.path.getsize('data/ours_ext/X.npz') < 100000:
    os.system('git lfs install; git lfs pull')
import xgboost; print('xgboost', xgboost.__version__, '| cwd:', os.getcwd())

In [ ]:
# Máquina onde está rodando (para reportar o custo de execução)
import platform, os, subprocess
MACHINE = {'platform': platform.platform(),
           'processor': platform.processor() or platform.machine(),
           'cpu_logical': os.cpu_count()}
try:
    import psutil
    MACHINE['cpu_physical'] = psutil.cpu_count(logical=False)
    MACHINE['ram_gb'] = round(psutil.virtual_memory().total / 1e9, 1)
except Exception:
    MACHINE['cpu_physical'] = None; MACHINE['ram_gb'] = None
try:
    MACHINE['gpu'] = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
        text=True).strip()
except Exception:
    MACHINE['gpu'] = None
print('Plataforma :', MACHINE['platform'])
print('Processador:', MACHINE['processor'])
print(f"CPU cores  : logicos = {MACHINE['cpu_logical']} | fisicos = {MACHINE['cpu_physical']}")
print('RAM        :', MACHINE['ram_gb'], 'GB')
print('GPU        :', MACHINE['gpu'] or 'nenhuma (CPU)')

In [ ]:
%matplotlib inline
import json, numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (classification_report, confusion_matrix, roc_curve, auc,
                             precision_recall_curve, average_precision_score)
from sklearn.preprocessing import label_binarize
from xgboost import XGBClassifier

CLASSES = ['normal', 'dos', 'fuzzy', 'mitm_single', 'mitm_multi']; N = len(CLASSES)
ROOT = '..' if os.path.basename(os.getcwd()) == 'notebooks' else '.'
DATA = os.path.join(ROOT, 'data', 'ours_ext')
RESULTS = os.path.join(ROOT, 'results'); os.makedirs(RESULTS, exist_ok=True)

# 16 features comportamentais (indices no X.npz de 21 colunas), sem cabecalho
CONTENT_EXT = list(range(12)) + [12, 13, 14, 16]
PARAMS = dict(objective='multi:softprob', num_class=N, n_estimators=300, max_depth=8,
              learning_rate=0.3, tree_method='hist', device='cuda',  # sem GPU: device='cpu'
              max_bin=256, n_jobs=-1, eval_metric='mlogloss')
# fronteiras dos 7 PCAPs (ordem de extracao) para o split temporal por arquivo
FILE_COUNTS = [2193802, 1864530, 2197113, 1304154, 2223650, 2412529, 2037576]
print('RESULTS ->', os.path.abspath(RESULTS))

In [ ]:
names = json.load(open(f'{DATA}/feature_names.json'))
Xall = np.load(f'{DATA}/X.npz')['a']
y = np.load(f'{DATA}/y_multi.npz')['a']
assert Xall.shape[0] == sum(FILE_COUNTS), (Xall.shape[0], sum(FILE_COUNTS))
X16 = Xall[:, CONTENT_EXT].copy(); del Xall
FNAMES = [names[i] for i in CONTENT_EXT]
print('16 features comportamentais:', FNAMES)

def temporal_split(frac=0.70):
    tr, te, start = [], [], 0
    for cnt in FILE_COUNTS:                      # primeiros 70% de CADA arquivo -> treino
        cut = start + int(cnt * frac)            # ultimos 30% (mais tarde no tempo) -> teste
        tr.append(np.arange(start, cut)); te.append(np.arange(cut, start + cnt)); start += cnt
    return np.concatenate(tr), np.concatenate(te)

itr, ite = temporal_split(); y_te = y[ite]
print(f'treino = {itr.size:,}   teste = {ite.size:,}')
print('teste por classe:', dict(zip(CLASSES, np.bincount(y_te, minlength=N).tolist())))

In [ ]:
# Modelo: 16 features comportamentais
clf = XGBClassifier(**PARAMS); clf.fit(X16[itr], y[itr])
proba = clf.predict_proba(X16[ite]); y_pred = proba.argmax(1)

rep = classification_report(y_te, y_pred, target_names=CLASSES, digits=4, output_dict=True)
macro_f1 = rep['macro avg']['f1-score']; acc = rep['accuracy']
print(classification_report(y_te, y_pred, target_names=CLASSES, digits=4))
print(f'macro-F1 = {macro_f1:.4f}   accuracy = {acc:.4f}')

In [ ]:
# Matriz de confusao (normalizada por linha) -> results/cm_temporal.png
cm = confusion_matrix(y_te, y_pred); cmn = cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(6.5, 5.5)); im = ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(N)); ax.set_yticks(range(N))
ax.set_xticklabels(CLASSES, rotation=45, ha='right'); ax.set_yticklabels(CLASSES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title('Confusion matrix (temporal split)')
for i in range(N):
    for j in range(N):
        ax.text(j, i, f'{cmn[i,j]*100:.1f}%\n({cm[i,j]:,})', ha='center', va='center',
                fontsize=7, color='white' if cmn[i,j] > 0.5 else 'black')
fig.colorbar(im, fraction=0.046, pad=0.04); plt.tight_layout()
fig.savefig(f'{RESULTS}/cm_temporal.png', dpi=200, bbox_inches='tight'); plt.show()

In [ ]:
# Curvas ROC e Precisao-Recall (one-vs-rest) -> results/roc_pr_temporal.png
yb = label_binarize(y_te, classes=range(N))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5)); roc_auc = {}; pr_ap = {}
for i, name in enumerate(CLASSES):
    fpr, tpr, _ = roc_curve(yb[:, i], proba[:, i]); a = auc(fpr, tpr); roc_auc[name] = float(a)
    s = max(len(fpr) // 2000, 1); ax1.plot(fpr[::s], tpr[::s], lw=1.6, label=f'{name} (AUC={a:.3f})')
    prec, rec, _ = precision_recall_curve(yb[:, i], proba[:, i])
    ap = average_precision_score(yb[:, i], proba[:, i]); pr_ap[name] = float(ap)
    s2 = max(len(rec) // 2000, 1); ax2.plot(rec[::s2], prec[::s2], lw=1.6, label=f'{name} (AP={ap:.3f})')
ax1.plot([0, 1], [0, 1], 'k--', lw=0.7); ax1.set_xlabel('False Positive Rate'); ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC (one-vs-rest, temporal)'); ax1.legend(fontsize=8, loc='lower right')
ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision'); ax2.set_title('Precision-Recall (one-vs-rest, temporal)')
ax2.legend(fontsize=8, loc='lower left'); plt.tight_layout()
fig.savefig(f'{RESULTS}/roc_pr_temporal.png', dpi=200, bbox_inches='tight'); plt.show()

In [ ]:
# Importancia das features (ganho) -> results/feature_importance.png
score = clf.get_booster().get_score(importance_type='gain')
imp = np.array([score.get(f'f{i}', 0.0) for i in range(len(CONTENT_EXT))])
order = np.argsort(imp)
fig, ax = plt.subplots(figsize=(7, 5.5))
ax.barh([FNAMES[i] for i in order], imp[order])
ax.set_xlabel('Importancia (gain, XGBoost)'); ax.set_title('Importancia das 16 features (split temporal)')
plt.tight_layout(); fig.savefig(f'{RESULTS}/feature_importance.png', dpi=200, bbox_inches='tight'); plt.show()
feat_imp = {FNAMES[i]: float(imp[i]) for i in order[::-1]}
for k, v in feat_imp.items():
    print(f'  {k:18} {v:10.2f}')

In [ ]:
# Tempo de INFERENCIA (por pacote)
import time
Xte = X16[ite]
_ = clf.predict_proba(Xte[:2000])                 # warm-up
t0 = time.perf_counter(); _ = clf.predict_proba(Xte); t1 = time.perf_counter()
inf_total = t1 - t0; n = Xte.shape[0]
inf_us = inf_total / n * 1e6
print(f'Inferencia ({PARAMS["device"]}): {n:,} pacotes em {inf_total:.3f}s '
      f'-> {inf_us:.2f} us/pacote  ({n/inf_total:,.0f} pkt/s)')

In [ ]:
# Tempo de EXTRACAO de features (requer PCAPs ~1.9GB; use False para pular)
import sys, time
MEASURE_EXTRACTION = True
EXTRACT_SAMPLE = 200_000                            # nº de pacotes cronometrados
ext_us = None
sys.path.insert(0, os.path.join(ROOT, 'src'))
try:
    try:
        import scapy  # noqa: F401
    except ImportError:
        os.system('pip install -q scapy')
    from extract import fit_bytemodels, PCAP_DIR_DEFAULT, BENIGN_FILE
    import extract_ext, labeler
    pcap_dir = os.path.join(ROOT, PCAP_DIR_DEFAULT); benign = os.path.join(pcap_dir, BENIGN_FILE)
    if not os.path.exists(benign) and MEASURE_EXTRACTION:
        print('baixando PCAPs (~1.9GB)...'); os.system(f'{sys.executable} scripts/download_pcaps.py')
    if os.path.exists(benign):
        models, benign_ips = fit_bytemodels(pcap_dir, max_packets=EXTRACT_SAMPLE)
        t0 = time.perf_counter()
        Xf, yf, atk = extract_ext.extract_file(benign, models, attack_type=labeler.NORMAL,
                                               benign_ips=benign_ips, max_packets=EXTRACT_SAMPLE)
        t1 = time.perf_counter(); ne = Xf.shape[0]; ext_us = (t1 - t0) / ne * 1e6
        print(f'Extracao: {ne:,} pacotes em {t1-t0:.2f}s -> {ext_us:.2f} us/pacote  ({ne/(t1-t0):,.0f} pkt/s)')
        print(f'Razao extracao/inferencia ~ {ext_us/inf_us:.1f}x (extracao domina o custo)')
    else:
        print('PCAPs ausentes -> extracao pulada. Rode scripts/download_pcaps.py para medir.')
except Exception as e:
    print('extracao pulada:', repr(e)[:200])

In [ ]:
# Ablacao: 12 features (nucleo, sem as 4 adicionais) vs as 16, mesmo split temporal
clf12 = XGBClassifier(**PARAMS); clf12.fit(X16[itr][:, :12], y[itr])
pred12 = clf12.predict(X16[ite][:, :12])
rep12 = classification_report(y_te, pred12, target_names=CLASSES, digits=4, output_dict=True)
macro12 = rep12['macro avg']['f1-score']
print(f'Ablacao (temporal) macro-F1:  12 features = {macro12:.4f}   |   16 features = {macro_f1:.4f}')
print('F1 por classe:')
for c in CLASSES:
    print(f'  {c:12}  12={rep12[c]["f1-score"]:.4f}   16={rep[c]["f1-score"]:.4f}')

In [ ]:
# Salva todas as metricas -> results/metrics_temporal.json
out = {
    'machine': MACHINE,
    'split': 'temporal_by_file_70_30',
    'params': {k: v for k, v in PARAMS.items()},
    'test_counts': dict(zip(CLASSES, np.bincount(y_te, minlength=N).tolist())),
    'model_16feat': {
        'macro_f1': float(macro_f1), 'accuracy': float(acc),
        'per_class': {c: {k: float(rep[c][k]) for k in ('precision', 'recall', 'f1-score')} for c in CLASSES},
        'roc_auc': roc_auc, 'pr_ap': pr_ap, 'feature_importance_gain': feat_imp,
    },
    'ablation_12feat': {'macro_f1': float(macro12),
                        'per_class_f1': {c: float(rep12[c]['f1-score']) for c in CLASSES}},
    'timing': {'inference_us_per_pkt': float(inf_us),
               'extraction_us_per_pkt': (float(ext_us) if ext_us else None),
               'extraction_over_inference': (float(ext_us / inf_us) if ext_us else None)},
}
json.dump(out, open(f'{RESULTS}/metrics_temporal.json', 'w'), indent=2)
print('salvo:', os.path.abspath(RESULTS))
print(json.dumps({'macro_f1_16': macro_f1, 'macro_f1_12': macro12, 'accuracy': acc,
                  'inf_us': round(inf_us, 2), 'ext_us': (round(ext_us, 2) if ext_us else None)}, indent=2))

## Leitura

- **macro-F1 (temporal) ~0,97**, com **recall macro ~0,996**: o detector praticamente não deixa ataque passar.
- **`mitm_multi`** é a única classe que cai — por **precisão** (falso alarme), não por *recall*: o MITM *relay*
  **republica notificação legítima**, a fronteira mais borrada frente ao `normal`.
- **Ablação:** com **12 features** o macro-F1 desaba (a confusão se concentra em `fuzzy` e `mitm_multi`);
  as **4 adicionais** recuperam essas classes — o gargalo era a **representação**, não o modelo.
- **Custo:** a **extração de features domina** o tempo por pacote frente à **inferência** (razão reportada acima).

Artefatos em `results/`: `cm_temporal.png`, `roc_pr_temporal.png`, `feature_importance.png`, `metrics_temporal.json`.
